In [1]:
!pip install pandas numpy scikit-learn

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

In [6]:
def load_dataset(file_path, target_column):

    df = pd.read_csv(file_path)

    X = df.drop(columns=[target_column])
    y = df[target_column]

    return X, y

In [7]:
def build_preprocessor(X):

    numeric_features = X.select_dtypes(include=['int64','float64']).columns
    categorical_features = X.select_dtypes(include=['object']).columns

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ])

    return preprocessor

In [8]:
def compare_models(preprocessor, X_train, X_test, y_train, y_test):

    models = {
        "Logistic Regression": LogisticRegression(max_iter=1000),
        "Decision Tree": DecisionTreeClassifier(),
        "Random Forest": RandomForestClassifier(),
        "SVM": SVC()
    }

    results = {}

    for name, model in models.items():

        pipe = Pipeline([
            ("preprocessing", preprocessor),
            ("classifier", model)
        ])

        pipe.fit(X_train, y_train)

        preds = pipe.predict(X_test)

        acc = accuracy_score(y_test, preds)

        results[name] = acc

        print(name, "Accuracy:", acc)

    return results

In [9]:
def tune_random_forest(preprocessor, X_train, y_train):

    pipe = Pipeline([
        ("preprocessing", preprocessor),
        ("classifier", RandomForestClassifier())
    ])

    param_grid = {
        "classifier__n_estimators":[100,200],
        "classifier__max_depth":[10,20,None],
        "classifier__min_samples_split":[2,5]
    }

    grid = GridSearchCV(
        pipe,
        param_grid,
        cv=5,
        scoring="accuracy",
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    print("Best Parameters:", grid.best_params_)

    return grid.best_estimator_

In [10]:
def run_pipeline(dataset_path, target_column):

    print("Loading dataset")

    X, y = load_dataset(dataset_path, target_column)

    preprocessor = build_preprocessor(X)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    print("Comparing Models")

    results = compare_models(
        preprocessor,
        X_train,
        X_test,
        y_train,
        y_test
    )

    best_model = max(results, key=results.get)

    print("Best Model:", best_model)

    print("Running Hyperparameter Tuning")

    tuned_model = tune_random_forest(
        preprocessor,
        X_train,
        y_train
    )

    preds = tuned_model.predict(X_test)

    print("Final Accuracy:", accuracy_score(y_test, preds))

    print(classification_report(y_test, preds))

In [ ]:
# Example usage

run_pipeline(
    dataset_path="your_dataset.csv",
    target_column="label"
)